# Create target list for the BGS Extension spare fiber program

# Setup:
Ensure that this notebook is running on the kernel with the legacy survey cutout service + dask docker image.

To set up the Dask cluster, run following cell in Jupyter terminal

`salloc -N 4 -n 512 -t 240 -C cpu -q interactive --image=biprateep/desi-dask:latest --account=desi`

and then 

`./launch_dask.sh` 

the `-n` argument controls the number of workers to be launched.

In [1]:
import os
import shutil
from pathlib import Path
from importlib import reload


import numpy as np
import pandas as pd
from astropy.table import Table, vstack, hstack
import fitsio
import dask
from dask.distributed import Client
import dask.dataframe

from desitarget.targets import encode_targetid
from desimodel.footprint import radec2pix
import matplotlib.pyplot as plt

Matplotlib created a temporary cache directory at /tmp/matplotlib-rhjsbfw5 because the default path (/homedir/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [3]:
# Read the scheduler file generated by the script above and connect your notebook to the client
scheduler_file = os.path.join(os.environ["SCRATCH"], "scheduler.json")
# scheduler_file = os.path.join(os.environ["CFS"],'desi','users','bid13', "scheduler.json")
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status"
client = Client(scheduler_file=scheduler_file)
client

<Client: 'tcp://10.249.1.92:8786' processes=512 threads=1024, memory=238.18 TiB>

In [4]:
cat_path = Path("/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/")
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"


### Dask delayed function to read and select required galaxies

In [5]:
@dask.delayed
def read_catalogs(paths, survey):
    dfs = list()
    for path in paths:    
        sweep = Table(fitsio.read(path))
        sweep.remove_column('DCHISQ')
        sweep["PHOTSYS"] = survey[0].upper()
        sweep["MYID"] = encode_targetid(objid = sweep["OBJID"],
                                        brickid = sweep["BRICKID"],
                                        release = sweep["RELEASE"])
        sweep["HEALPIX"] = radec2pix(nside = 64, ra = sweep["RA"], dec = sweep["DEC"])
        
        
        GFLUX = sweep["FLUX_G"] / sweep["MW_TRANSMISSION_G"]
        RFLUX = sweep["FLUX_R"] / sweep["MW_TRANSMISSION_R"]
        ZFLUX = sweep["FLUX_Z"] / sweep["MW_TRANSMISSION_Z"]
        W1FLUX = sweep["FLUX_W1"] / sweep["MW_TRANSMISSION_W1"]
        W1SNR = sweep["FLUX_W1"] * np.sqrt(sweep["FLUX_IVAR_W1"])
        G = sweep["GAIA_PHOT_G_MEAN_MAG"]
        Grr = sweep["GAIA_PHOT_G_MEAN_MAG"] - 22.5 + 2.5*np.log10(sweep["FLUX_R"])
        RFIBREFLUX = sweep["FIBERFLUX_R"]/ sweep["MW_TRANSMISSION_R"]
        rmag = 22.5 - 2.5*np.log10(np.clip(RFLUX, 1e-16, None))
        zmag = 22.5 - 2.5*np.log10(np.clip(ZFLUX, 1e-16, None))
        gmag = 22.5 - 2.5*np.log10(np.clip(GFLUX, 1e-16, None))
        w1mag = 22.5 - 2.5*np.log10(np.clip(W1FLUX, 1e-16, None))
        rfibmag = 22.5 - 2.5*np.log10(np.clip(RFIBREFLUX, 1e-16, None))
        rfibtotmag = 22.5 - 2.5*np.log10(np.clip(sweep["FIBERTOTFLUX_R"], 1e-16, None))
        
   
    
        #Valid photometry
        QC =  (sweep["FLUX_IVAR_G"] > 0) & (sweep["FLUX_IVAR_R"] > 0) & (sweep["FLUX_IVAR_Z"] > 0)
    
        
        
        #color range sanity
        QC &= ((gmag-rmag) > -1) & ((gmag-rmag) < 4)
        QC &= ((rmag-zmag) > -1) & ((rmag-zmag) < 4)
        
        # Atleast one observation
        QC &= (sweep["NOBS_G"] > 0) & (sweep["NOBS_R"] > 0) & (sweep["NOBS_Z"] > 0) 
        
        #Star galaxy separation
        #Will apply the TYPE!= PSF cut later on for easier reconstruction of parent sample
        # GALAXY = ((zmag - w1mag) > 0.8 * (rmag - zmag) - 1) | ((rmag - zmag) < 0.6) # Discarding in favor of PSF, W1 detection may throw out star forming galaxies
        # GALAXY = sweep["TYPE"] != "PSF"
        
    
        

        
        #Bright star and globular cluster masks        
        MASKBITS = ((sweep["MASKBITS"] & 2**1) == 0) #touches bright star
        MASKBITS &= ((sweep["MASKBITS"] & 2**13) == 0) #touches globular cluster
        #(should we mask SGA?)
        MASKBITS &= ((sweep["MASKBITS"] & 2**12) == 0) #touches SGA
        
        
        
        
        MAGLIM = (rmag >=19.5) & (rmag <= 21)
        
        
        bgs_mask = (  MASKBITS & QC & MAGLIM )    

        dfs.append(sweep[bgs_mask].to_pandas())
    return pd.concat(dfs)

In [6]:
# Chunk size is the number of files each worker opens
chunksize = 10
parts = list()

survey = "south" # choose from `north` or `south
sweep_path = cat_path / survey / "sweep" / "9.0"
sweep_files = list(sweep_path.glob("sweep*"))
for i in range(0, len(sweep_files), chunksize):
    parts.append(read_catalogs(sweep_files[i:i+chunksize], survey))

survey = "north" # choose from `north` or `south
sweep_path = cat_path / survey / "sweep" / "9.0"
sweep_files = list(sweep_path.glob("sweep*"))
for i in range(0, len(sweep_files), chunksize):
    parts.append(read_catalogs(sweep_files[i:i+chunksize], survey))

In [7]:
dd = dask.dataframe.from_delayed(parts)
dd = dd.set_index("MYID")
dd = dd.repartition(npartitions=2048)
dd = dd.persist()

In [8]:
filepath = dest_path / f"bgs_ext_target_list_ALL.parquet"
dd.to_parquet(filepath, compression='gzip', engine='pyarrow',overwrite=True,
              write_metadata_file=True)

In [9]:
print("Total number of Objects:", len(dd)/1e6)

Total number of Objects: 108.965525


### READ and save main survey TARGETIDs

In [1]:
from astropy.table import Table, hstack, vstack
import fitsio
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import os
import pandas as pd

In [2]:
main_survey_targets = Path("/global/cfs/cdirs/desi/survey/ops/surveyops/trunk/mtl/main/")
main_survey_programs = ["dark", "bright", "dark1b", "bright1b"]
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"

In [3]:
def read_target_list(fp):
    targets = Table.read(fp, format="ascii.ecsv")
    return targets[["RA","DEC", "DESI_TARGET", "TARGETID"]]

In [4]:
all_targets = []
for program in main_survey_programs:
    print(f"processing {program} program")

    table_list = Parallel(n_jobs=100)(
    delayed(read_target_list)(fp) 
    for fp in tqdm((main_survey_targets / program ).glob("*.ecsv")))
    all_targets.append(vstack(table_list, metadata_conflicts="silent").to_pandas())

processing dark program


6399it [00:22, 289.54it/s]


processing bright program


6462it [00:17, 360.18it/s]


processing dark1b program


6290it [00:03, 1999.24it/s]


processing bright1b program


1797it [00:00, 2155.21it/s]


In [5]:
all_targets = pd.concat(all_targets)
all_targets.to_parquet( dest_path / "all_main_survey_targets.parquet")
print("Total number of targets: ", len(all_targets)/1e6)

Total number of targets:  224.902714


### Resample the data and remove main survey targets

In [1]:
from pathlib import Path
import os
import pandas as pd
import fitsio
from astropy.table import Table
import numpy as np
from desitarget.targetmask import desi_mask
import pandas as pd

In [2]:
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"

my_targets = pd.read_parquet(dest_path / f"bgs_ext_target_list_ALL.parquet",
                     columns = ["MYID",'RA','DEC','TYPE', 'RELEASE', 'BRICKID', 'OBJID', 'PMRA', 'PMDEC', 'REF_EPOCH', 'FLUX_G', 'FLUX_R', 'FLUX_Z'])
my_targets = my_targets.reset_index()

desi_targets = pd.read_parquet(dest_path / "all_main_survey_targets.parquet")

In [3]:
desi_tgt = desi_targets["DESI_TARGET"]

is_bgs  = (desi_tgt & desi_mask.BGS_ANY != 0)   #- instead of 2**60
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_bgs])
print(f"Overlap Fraction with BGS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lrg  = (desi_tgt & desi_mask.LRG != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_lrg])
print(f"Overlap Fraction with LRG:{overlaps.sum()*100/len(overlaps)}")

is_elg  = (desi_tgt & desi_mask.ELG != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_elg])
print(f"Overlap Fraction with ELG:{overlaps.sum()*100/len(overlaps)}")

is_qso  = (desi_tgt & desi_mask.QSO != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_qso])
print(f"Overlap Fraction with QSO:{overlaps.sum()*100/len(overlaps)}")

is_mws  = (desi_tgt & desi_mask.MWS_ANY != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_mws])
print(f"Overlap Fraction with MWS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lge = (desi_tgt & desi_mask.LGE != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_lge])
print(f"Overlap Fraction with LGE:{overlaps.sum()*100/len(overlaps)}")


is_psf = (my_targets['TYPE'] == "PSF")
print(f"Fraction of PSF objects:{is_psf.sum()*100/len(is_psf)}")

Overlap Fraction with BGS_ANY:9.714603770320934
Overlap Fraction with LRG:3.1749986979826876
Overlap Fraction with ELG:0.4934276230945521
Overlap Fraction with QSO:1.3082890207705602
Overlap Fraction with MWS_ANY:3.042266808699357
Overlap Fraction with LGE:1.9351542609462946
Fraction of PSF objects:43.161135597703954


In [4]:
# remove PSF type objects
my_targets = my_targets[my_targets["TYPE"]!="PSF"]
len(my_targets)

61934767

In [5]:
rng = np.random.default_rng(299792458)

resampled_my_targets = my_targets.sample(frac = 0.05, random_state=rng)

In [6]:
resampled_my_targets.to_parquet(dest_path / "bgs_ext_target_list_DOWNSAMPLED.parquet")

In [7]:
is_bgs  = (desi_tgt & desi_mask.BGS_ANY != 0)   #- instead of 2**60
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_bgs])
print(f"Overlap Number with BGS_ANY:{overlaps.sum()}")
print(f"Overlap Fraction with BGS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lrg  = (desi_tgt & desi_mask.LRG != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_lrg])
print(f"Overlap Number with LRG:{overlaps.sum()}")
print(f"Overlap Fraction with LRG:{overlaps.sum()*100/len(overlaps)}")

is_elg  = (desi_tgt & desi_mask.ELG != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_elg])
print(f"Overlap Number with ELG:{overlaps.sum()}")
print(f"Overlap Fraction with ELG:{overlaps.sum()*100/len(overlaps)}")

is_qso  = (desi_tgt & desi_mask.QSO != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_qso])
print(f"Overlap Number with QSO:{overlaps.sum()}")
print(f"Overlap Fraction with QSO:{overlaps.sum()*100/len(overlaps)}")

is_mws  = (desi_tgt & desi_mask.MWS_ANY != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_mws])
print(f"Overlap Number with MWS_ANY:{overlaps.sum()}")
print(f"Overlap Fraction with MWS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lge = (desi_tgt & desi_mask.LGE != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_lge])
print(f"Overlap Number with LGE:{overlaps.sum()}")
print(f"Overlap Fraction with LGE:{overlaps.sum()*100/len(overlaps)}")



Overlap Number with BGS_ANY:526136
Overlap Fraction with BGS_ANY:16.990006904039024
Overlap Number with LRG:167503
Overlap Fraction with LRG:5.409014259520824
Overlap Number with ELG:8461
Overlap Fraction with ELG:0.2732229849603034
Overlap Number with QSO:0
Overlap Fraction with QSO:0.0
Overlap Number with MWS_ANY:368
Overlap Fraction with MWS_ANY:0.011883472221414922
Overlap Number with LGE:103520
Overlap Fraction with LGE:3.342872403154545


In [8]:
# remove BGS_ANY
mask = ~np.isin(resampled_my_targets["MYID"],desi_targets["TARGETID"][is_bgs])
final_my_targets = resampled_my_targets[mask]
print(f"{(~mask).sum()} BGS_ANY removed")

# remove LRG
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_lrg])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} LRG removed")

# remove ELG
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_elg])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} ELG removed")

# remove QSO
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_qso])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} QSO removed")

# remove LGE
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_lge])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} LGE removed")

526136 BGS_ANY removed
136633 LRG removed
6822 ELG removed
0 QSO removed
87220 LGE removed


In [9]:
print(f"Final target List length:{len(final_my_targets)/1e6}")

Final target List length:2.339927


In [10]:
final_my_targets.to_parquet(dest_path / "bgs_ext_target_list.parquet")

# Comply with the pipeline format

In [11]:
final_my_targets = Table.from_pandas(final_my_targets)
final_my_targets["OVERRIDE"] = False

In [12]:
final_my_targets.dtype

dtype([('MYID', '<i8'), ('RA', '<f8'), ('DEC', '<f8'), ('TYPE', '<U3'), ('RELEASE', '<i2'), ('BRICKID', '<i4'), ('OBJID', '<i4'), ('PMRA', '<f4'), ('PMDEC', '<f4'), ('REF_EPOCH', '<f4'), ('FLUX_G', '<f4'), ('FLUX_R', '<f4'), ('FLUX_Z', '<f4'), ('OVERRIDE', '?')])

In [24]:
arr = final_my_targets.as_array()
flipped_byte_order_cat = Table(arr.view(arr.dtype.newbyteorder()).byteswap())
print(flipped_byte_order_cat.dtype)

[('MYID', '>i8'), ('RA', '>f8'), ('DEC', '>f8'), ('TYPE', '>U3'), ('RELEASE', '>i2'), ('BRICKID', '>i4'), ('OBJID', '>i4'), ('PMRA', '>f4'), ('PMDEC', '>f4'), ('REF_EPOCH', '>f4'), ('FLUX_G', '>f4'), ('FLUX_R', '>f4'), ('FLUX_Z', '>f4'), ('OVERRIDE', '?')]


In [25]:
flipped_byte_order_cat

MYID,RA,DEC,TYPE,RELEASE,BRICKID,OBJID,PMRA,PMDEC,REF_EPOCH,FLUX_G,FLUX_R,FLUX_Z,OVERRIDE
int64,float64,float64,str3,int16,int32,int32,float32,float32,float32,float32,float32,float32,bool
39628135674678097,244.09070906098057,14.58856882228854,EXP,9010,414040,2897,0.0,0.0,0.0,1.5771352,4.996265,11.535962,False
39628453892327608,147.9163118594767,28.84742225931509,REX,9010,489909,2232,0.0,0.0,0.0,0.8724631,3.986708,9.013914,False
39627969714461251,249.95749672045375,7.506745620619057,EXP,9010,374472,6723,0.0,0.0,0.0,1.3748467,3.4763455,6.161241,False
39626570842776293,88.86075191884376,-61.272415355585835,DEV,9010,40955,8933,0.0,0.0,0.0,1.8681482,4.704737,8.036269,False
39633353321742530,290.95468004501015,57.14912359240768,EXP,9011,609448,194,0.0,0.0,0.0,2.319715,4.093439,6.259249,False
39628446158034163,344.11153653405967,28.279506209829748,REX,9010,488065,5363,0.0,0.0,0.0,1.7078702,4.377139,7.22153,False
39628532808157183,204.03082212677015,32.5413500917522,DEV,9010,508724,2047,0.0,0.0,0.0,4.6387625,13.314425,25.789927,False
39633545802548551,149.5561023757269,78.55275586179293,EXP,9011,655339,1351,0.0,0.0,0.0,2.1251166,5.0301533,7.3019514,False
39627658857810594,51.249534929537404,-5.209404887232112,DEV,9010,300358,2722,0.0,0.0,0.0,1.3357714,4.2401867,11.593024,False


In [26]:
flipped_byte_order_cat.write("/global/cfs/cdirs/desi/target/proposals/spare_proposals_Y3/indata/BGS_EXT.fits", overwrite=True)

In [27]:
from desitarget.secondary import check_file_format
check_file_format("/global/cfs/cdirs/desi/target/proposals/spare_proposals_Y3/indata/BGS_EXT.fits")

Matplotlib created a temporary cache directory at /tmp/matplotlib-4o6f51m7 because the default path (/homedir/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


True

### Optimize the Star Galaxy separation

In [ ]:
import os
from pathlib import Path
import pandas as pd
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
import mpl_scatter_density
# from secondary import match_to_main_survey

In [ ]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [ ]:
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"
cat = pd.read_parquet(dest_path / f"spare_fiber_photo_z_target_list_TEST",
                     columns = ["FLUX_R","FLUX_Z","FLUX_W1","FLUX_G",
                               "MW_TRANSMISSION_R", "MW_TRANSMISSION_Z", "MW_TRANSMISSION_G", "MW_TRANSMISSION_W1", "TYPE", "RA", "DEC"])

In [ ]:
cat = cat.sample(n=100000)
RFLUX = cat["FLUX_R"] / cat["MW_TRANSMISSION_R"]
ZFLUX = cat["FLUX_Z"] / cat["MW_TRANSMISSION_Z"]
ZFLUX = cat["FLUX_Z"] / cat["MW_TRANSMISSION_Z"]
GFLUX = cat["FLUX_G"] / cat["MW_TRANSMISSION_G"]
W1FLUX = cat["FLUX_W1"] / cat["MW_TRANSMISSION_W1"]
rmag = 22.5 - 2.5*np.log10(np.clip(RFLUX, 1e-16, None))
zmag = 22.5 - 2.5*np.log10(np.clip(ZFLUX, 1e-16, None))
gmag = 22.5 - 2.5*np.log10(np.clip(GFLUX, 1e-16, None))
w1mag = 22.5 - 2.5*np.log10(np.clip(W1FLUX, 1e-16, None))
        

In [ ]:
def cmap_white(cmap_name):
    """Returns a colormap with white as the lowest value color."""
    import numpy as np
    try:
        from matplotlib import cm
        from matplotlib.colors import ListedColormap
        cmap = cm.get_cmap(cmap_name, 256)
    except ValueError:
        import seaborn as sns
        cmap = sns.color_palette("flare", as_cmap=True)
    newcolors = cmap(np.linspace(0, 1, 256))
    white = np.array([1, 1, 1, 0])
    newcolors[:1, :] = white
    cmap_white = ListedColormap(newcolors)
    return cmap_white

In [ ]:
cmap = cmap_white("viridis")
xmin = 0.6

x = np.linspace(xmin,3)


y = 0.8 * x -0.6

y1 = 0.8* x - 1


is_psf = (cat["TYPE"]=="PSF")
is_galaxy = ((zmag - w1mag) > 0.8 * (rmag - zmag) - 1) | ((rmag - zmag) < 0.6)

fig = plt.figure(figsize=(COLUMN_WIDTH, COLUMN_WIDTH))
ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
ax.scatter_density(rmag-zmag, zmag-w1mag,cmap=cmap,dpi=200,downres_factor=2,alpha=1)
ax.scatter((rmag-zmag)[is_psf], (zmag-w1mag)[is_psf], c="k",marker=".", s = 0.01,alpha=0.5)
ax.set_xlim(-0.1,3)
ax.set_ylim(-3,2.5)


# ax.plot(x,y,"--k",label="LRG star-galaxy")
ax.plot(x,y1,"--k",alpha=0.5,label="NEW star-galaxy")
ax.axvline(xmin,ls="--",c="k",alpha=0.5)
ax.set_xlabel("r-z")
ax.set_ylabel("z-w1")
# fig.legend()

In [ ]:
cmap = cmap_white("viridis")
xmin = 0.6

x = np.linspace(xmin,3)


y = 0.8 * x -0.6

y1 = 0.8* x - 1


# is_psf = (cat["TYPE"]=="PSF")
# is_galaxy = ((zmag - w1mag) > 0.8 * (rmag - zmag) - 1) | ((rmag - zmag) < 0.6)

fig = plt.figure(figsize=(COLUMN_WIDTH, COLUMN_WIDTH))
ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
ax.scatter_density(rmag-zmag, gmag-rmag,cmap=cmap,dpi=200,downres_factor=2,alpha=1)
# ax.scatter((rmag-zmag)[is_psf], (zmag-w1mag)[is_psf], c="k",marker=".", s = 0.01,alpha=0.5)
# ax.set_xlim(-0.1,3)
# ax.set_ylim(-3,2.5)


# ax.plot(x,y,"--k",label="LRG star-galaxy")
# ax.plot(x,y1,"--k",alpha=0.5,label="NEW star-galaxy")
# ax.axvline(xmin,ls="--",c="k",alpha=0.5)
ax.set_xlabel("r-z")
ax.set_ylabel("g-r")
# fig.legend()